In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
uploaded = files.upload()
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()

Saving tariff_master.csv to tariff_master.csv
Saving substation_master.csv to substation_master.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_master.csv to meter_master.csv
Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv


In [3]:
def load_csv(filename):
  return spark.read.csv(filename, header=True, inferSchema=True)
readings_df = load_csv("meter_readings_90000.csv")

In [4]:
readings_df.printSchema()

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)



In [5]:
print("Total Rows:", readings_df.count())

Total Rows: 90000


In [6]:
readings_df.show(10, truncate=False)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0000001 |M0082   |2026-04-01 02:00:00|22.53     |213.1    |0.939       |0             |Valid         |SmartMeter |36.2                 |OP029      |
|RD0000002 |M0020   |2026-01-31 06:00:00|2.97      |227.4    |0.845       |0             |Valid         |SmartMeter |41.2                 |OP025      |
|RD0000003 |M0054   |2026-04-11 10:15:00|8.61      |239.3    |0.879       |0             |Estimated     |SmartMeter |20.9                 |OP007      |
|RD0000004 |M0111   |2026-03-14 00:00:00|1.87      |246.5    |0.831       |0            

In [7]:
selected_df = readings_df.select(
    "reading_id",
    "meter_id",
    "reading_timestamp",
    "energy_kwh",
    "reading_status"
)
selected_df.show(10)

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
| RD0000006|   M0139|2026-02-20 02:15:00|      0.36|         Valid|
| RD0000007|   M0142|2026-02-01 05:30:00|     79.47|         Valid|
| RD0000008|   M0126|2026-03-01 20:45:00|     28.35|      Rejected|
| RD0000009|   M0005|2026-05-03 10:30:00|      1.96|     Estimated|
| RD0000010|   M0052|2026-04-10 05:45:00|     22.52|         Valid|
+----------+--------+-------------------+----------+--------------+
only showing top 10 rows


In [8]:
renamed_df = readings_df.withColumnRenamed(
    "source_type",
    "collection_mode"
)
renamed_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|collection_mode|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid|     SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid|     SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated|     SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|  

In [9]:
filtered_df = readings_df.filter(
    (F.col("reading_status") == "Valid") &
    (F.col("energy_kwh") > 20)
)
sorted_df = filtered_df.orderBy(
    F.col("energy_kwh").desc()
)
sorted_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             0|         Valid| SmartMeter|                 20.1|      OP021|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             

In [10]:
energy_band_df = readings_df.withColumn(
    "energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High")
)
energy_band_df.select(
    "reading_id",
    "energy_kwh",
    "energy_band"
).show(10)

+----------+----------+-----------+
|reading_id|energy_kwh|energy_band|
+----------+----------+-----------+
| RD0000001|     22.53|     Medium|
| RD0000002|      2.97|        Low|
| RD0000003|      8.61|     Medium|
| RD0000004|      1.87|        Low|
| RD0000005|      4.39|        Low|
| RD0000006|      0.36|        Low|
| RD0000007|     79.47|       High|
| RD0000008|     28.35|       High|
| RD0000009|      1.96|        Low|
| RD0000010|     22.52|     Medium|
+----------+----------+-----------+
only showing top 10 rows


In [11]:
null_counts_df = readings_df.select(
    F.count(
        F.when(F.col("energy_kwh").isNull(), 1)
    ).alias("energy_null_count"),

    F.count(
        F.when(F.col("voltage_v").isNull(), 1)
    ).alias("voltage_null_count"),

    F.count(
        F.when(F.col("power_factor").isNull(), 1)
    ).alias("power_factor_null_count"),

    F.count(
        F.when(F.col("operator_id").isNull(), 1)
    ).alias("operator_null_count")
)
null_counts_df.show()

+-----------------+------------------+-----------------------+-------------------+
|energy_null_count|voltage_null_count|power_factor_null_count|operator_null_count|
+-----------------+------------------+-----------------------+-------------------+
|             1620|              2070|                   1440|               5400|
+-----------------+------------------+-----------------------+-------------------+



In [12]:
filled_df = readings_df.fillna({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})
filled_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|       0.831|             

In [13]:
original_count = readings_df.count()
cleaned_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() &
    F.col("voltage_v").isNotNull()
)
cleaned_count = cleaned_df.count()
removed_count = original_count - cleaned_count
print("Original Rows:", original_count)
print("Cleaned Rows:", cleaned_count)
print("Removed Rows:", removed_count)

Original Rows: 90000
Cleaned Rows: 86349
Removed Rows: 3651


In [15]:
meter_df = load_csv("meter_master.csv")
meter_join_df = readings_df.join(
    meter_df,
    on="meter_id",
    how="inner"
)
valid_energy_site_df = meter_join_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "site_type"
).agg(
    F.sum("energy_kwh").alias("total_energy_kwh")
)
valid_energy_site_df.show()

+---------------+-----------------+
|      site_type| total_energy_kwh|
+---------------+-----------------+
|Public Facility|        138870.41|
|    Residential|100863.2900000004|
|     Industrial|710317.4100000021|
|     Commercial|451525.1799999996|
+---------------+-----------------+



In [16]:
substation_df = load_csv("substation_master.csv")
substation_join_df = meter_join_df.join(
    substation_df,
    on="substation_id",
    how="inner"
)
valid_region_city_df = substation_join_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "region",
    "city"
).agg(
    F.sum("energy_kwh").alias("total_valid_energy")
)
valid_region_city_df.show()


+-------+-------------+------------------+
| region|         city|total_valid_energy|
+-------+-------------+------------------+
|Central|       Indore|176503.91000000012|
|   West|       Mumbai| 52116.20999999999|
|  South|    Hyderabad|51937.789999999884|
|  South|Visakhapatnam|205015.82999999932|
|   West|    Ahmedabad| 138627.7400000004|
|  North|      Lucknow| 142610.3999999998|
|  South|    Bengaluru| 73787.59999999998|
|  North|       Jaipur| 47863.15999999996|
|  South|        Kochi| 204272.4600000002|
|  North|        Delhi| 94234.89000000014|
|  South|      Chennai| 51829.05999999996|
|   West|         Pune|162777.23999999996|
+-------+-------------+------------------+



In [18]:
tariff_df = load_csv("tariff_master.csv")
tariff_join_df = readings_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).join(
    tariff_df,
    on="tariff_id",
    how="inner"
)
energy_charge_df = tariff_join_df.filter(
    (F.col("reading_status") != "Rejected") &
    (F.col("energy_kwh").isNotNull())
).withColumn(
    "energy_charge_inr",
    F.col("energy_kwh") * F.col("rate_per_kwh_inr")
)
energy_charge_df.select(
    "reading_id",
    "meter_id",
    "energy_kwh",
    "rate_per_kwh_inr",
    "energy_charge_inr"
).show(10)

+----------+--------+----------+----------------+------------------+
|reading_id|meter_id|energy_kwh|rate_per_kwh_inr| energy_charge_inr|
+----------+--------+----------+----------------+------------------+
| RD0000001|   M0082|     22.53|             6.7|150.95100000000002|
| RD0000002|   M0020|      2.97|             6.4|19.008000000000003|
| RD0000003|   M0054|      8.61|             8.2| 70.60199999999999|
| RD0000004|   M0111|      1.87|             5.1| 9.536999999999999|
| RD0000005|   M0135|      4.39|             8.2|            35.998|
| RD0000006|   M0139|      0.36|             6.4|             2.304|
| RD0000007|   M0142|     79.47|             9.1| 723.1769999999999|
| RD0000009|   M0005|      1.96|             5.1| 9.995999999999999|
| RD0000010|   M0052|     22.52|             6.7|150.88400000000001|
| RD0000011|   M0137|     10.51|             6.7|            70.417|
+----------+--------+----------+----------------+------------------+
only showing top 10 rows


In [19]:
maintenance_df = load_csv("maintenance_tickets_1600.csv")
maintenance_join_df = maintenance_df.join(
    meter_df,
    on="meter_id",
    how="inner"
)
maintenance_summary_df = maintenance_join_df.groupBy(
    "site_type",
    "severity"
).agg(
    F.count("ticket_id").alias("ticket_count"),
    F.avg("maintenance_cost_inr").alias(
        "average_maintenance_cost"
    )
)
maintenance_summary_df.show()

+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|average_maintenance_cost|
+---------------+--------+------------+------------------------+
|    Residential|Critical|          41|      11875.782702702705|
|     Industrial|  Medium|         106|      12043.898526315788|
|     Commercial|Critical|          37|      13424.726216216219|
|     Commercial|     Low|         166|      12223.517692307689|
|Public Facility|    High|          48|      12905.279347826086|
|    Residential|     Low|         194|       12531.90201117318|
|Public Facility|  Medium|          97|      13050.628421052634|
|     Industrial|Critical|          17|      12296.198000000002|
|Public Facility|Critical|          15|       8986.843571428572|
|     Industrial|    High|          63|      12838.869344262299|
|    Residential|    High|         156|      11047.967122302161|
|Public Facility|     Low|          60|      11195.929464285715|
|     Commercial|  Medium

In [20]:
substation_analysis_df = readings_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).groupBy(
    "substation_id"
).agg(
    F.avg("voltage_v").alias("average_voltage"),
    F.sum("outage_minutes").alias("total_outage_minutes")
)
top_five_substations_df = substation_analysis_df.orderBy(
    F.col("total_outage_minutes").desc()
).limit(5)
top_five_substations_df.show()

+-------------+------------------+--------------------+
|substation_id|   average_voltage|total_outage_minutes|
+-------------+------------------+--------------------+
|        SUB10|230.04010567581344|               91224|
|        SUB09|230.04586272640606|               81734|
|        SUB04| 230.0558687466385|               75872|
|        SUB08|229.88012557077673|               75581|
|        SUB03|230.03094896227708|               61662|
+-------------+------------------+--------------------+



In [21]:
status_count_df = readings_df.groupBy(
    "source_type",
    "reading_status"
).agg(
    F.count("*").alias("reading_count")
).orderBy(
    F.col("reading_count").desc()
)

status_count_df.show()

+-----------+--------------+-------------+
|source_type|reading_status|reading_count|
+-----------+--------------+-------------+
| SmartMeter|         Valid|        65521|
| SmartMeter|     Estimated|        10443|
|     Manual|         Valid|         8965|
| SmartMeter|      Rejected|         3202|
|     Manual|     Estimated|         1405|
|     Manual|      Rejected|          464|
+-----------+--------------+-------------+



In [22]:
timestamp_df = readings_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
)
monthly_energy_df = timestamp_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).join(
    substation_df,
    on="substation_id",
    how="inner"
).filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "year",
    F.year("reading_timestamp")
).withColumn(
    "month",
    F.month("reading_timestamp")
).groupBy(
    "region",
    "year",
    "month"
).agg(
    F.sum("energy_kwh").alias("monthly_valid_energy")
)
monthly_energy_df.orderBy(
    "region", "year", "month"
).show()

+-------+----+-----+--------------------+
| region|year|month|monthly_valid_energy|
+-------+----+-----+--------------------+
|Central|2026|    1|   31475.57999999999|
|Central|2026|    2|  28161.410000000007|
|Central|2026|    3|            29805.85|
|Central|2026|    4|  29475.339999999982|
|Central|2026|    5|  30048.879999999986|
|Central|2026|    6|   27536.84999999999|
|  North|2026|    1|  48849.309999999925|
|  North|2026|    2|  45444.249999999985|
|  North|2026|    3|   51094.61000000006|
|  North|2026|    4|   43451.02000000005|
|  North|2026|    5|   48796.93000000004|
|  North|2026|    6|   47072.32999999992|
|  South|2026|    1|   97547.38999999987|
|  South|2026|    2|   90864.80000000006|
|  South|2026|    3|  102258.77000000005|
|  South|2026|    4|   97266.94000000005|
|  South|2026|    5|   101204.8699999999|
|  South|2026|    6|   97699.97000000002|
|   West|2026|    1|   59967.56999999997|
|   West|2026|    2|   55807.89999999989|
+-------+----+-----+--------------

In [23]:
meter_energy_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
).join(
    meter_df.select("meter_id", "substation_id"),
    on="meter_id",
    how="inner"
)
rank_window = Window.partitionBy(
    "substation_id"
).orderBy(
    F.col("total_energy").desc()
)
ranked_meters_df = meter_energy_df.withColumn(
    "meter_rank",
    F.dense_rank().over(rank_window)
)
top_three_meters_df = ranked_meters_df.filter(
    F.col("meter_rank") <= 3
)
top_three_meters_df.show()

+--------+------------------+-------------+----------+
|meter_id|      total_energy|substation_id|meter_rank|
+--------+------------------+-------------+----------+
|   M0130|           9364.98|        SUB01|         1|
|   M0110| 9259.750000000002|        SUB01|         2|
|   M0068| 8832.580000000005|        SUB01|         3|
|   M0071|          29242.04|        SUB02|         1|
|   M0145|28641.319999999992|        SUB02|         2|
|   M0133| 7060.140000000003|        SUB02|         3|
|   M0096| 9632.740000000003|        SUB03|         1|
|   M0080| 9525.940000000002|        SUB03|         2|
|   M0073| 9369.789999999995|        SUB03|         3|
|   M0038|          29505.88|        SUB04|         1|
|   M0099|29216.319999999996|        SUB04|         2|
|   M0098|28613.460000000014|        SUB04|         3|
|   M0094| 9659.639999999996|        SUB05|         1|
|   M0050| 9570.859999999997|        SUB05|         2|
|   M0008| 9524.780000000004|        SUB05|         3|
|   M0126|

In [24]:
cumulative_window = Window.partitionBy(
    "meter_id"
).orderBy(
    F.col("reading_timestamp"),
    F.col("reading_id")
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)
cumulative_energy_df = timestamp_df.filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "cumulative_energy",
    F.sum("energy_kwh").over(cumulative_window)
)
cumulative_energy_df.select(
    "meter_id",
    "reading_timestamp",
    "reading_id",
    "energy_kwh",
    "cumulative_energy"
).show(20)

+--------+-------------------+----------+----------+------------------+
|meter_id|  reading_timestamp|reading_id|energy_kwh| cumulative_energy|
+--------+-------------------+----------+----------+------------------+
|   M0001|2026-01-01 01:15:00| RD0083587|     22.27|             22.27|
|   M0001|2026-01-01 03:30:00| RD0086275|      4.05|             26.32|
|   M0001|2026-01-01 09:45:00| RD0088204|     11.28|              37.6|
|   M0001|2026-01-02 04:45:00| RD0005149|     11.76|             49.36|
|   M0001|2026-01-02 08:15:00| RD0035530|     10.06|             59.42|
|   M0001|2026-01-02 18:30:00| RD0070104|     15.83|             75.25|
|   M0001|2026-01-03 00:30:00| RD0006677|     17.06|             92.31|
|   M0001|2026-01-03 06:00:00| RD0021042|     17.14|            109.45|
|   M0001|2026-01-03 08:30:00| RD0032205|     20.46|            129.91|
|   M0001|2026-01-03 18:15:00| RD0074606|      20.4|            150.31|
|   M0001|2026-01-04 19:30:00| RD0019547|      6.84|            

In [26]:
extra_readings_df = load_csv("meter_readings_extra_12000.csv")
main_count = readings_df.count()
extra_count = extra_readings_df.count()
combined_df = readings_df.unionByName(
    extra_readings_df
)
combined_count = combined_df.count()
print("Main Readings:", main_count)
print("Extra Readings:", extra_count)
print("Expected Combined Rows:", main_count + extra_count)
print("Actual Combined Rows:", combined_count)

Main Readings: 90000
Extra Readings: 12000
Expected Combined Rows: 102000
Actual Combined Rows: 102000


In [27]:
duplicate_readings_df = combined_df.groupBy(
    "reading_id"
).agg(
    F.count("*").alias("duplicate_count")
).filter(
    F.col("duplicate_count") > 1
)
print("Duplicate Reading IDs:")
duplicate_readings_df.show()
print("Repeated ID Count:", duplicate_readings_df.count())
unique_readings_df = combined_df.dropDuplicates(
    ["reading_id"]
)
print("Rows Before Deduplication:", combined_count)
print("Unique Reading Count:", unique_readings_df.count())

Duplicate Reading IDs:
+----------+---------------+
|reading_id|duplicate_count|
+----------+---------------+
| RD0025848|              2|
| RD0045533|              2|
| RD0064921|              2|
| RD0056575|              2|
| RD0058815|              2|
| RD0022131|              2|
| RD0029598|              2|
| RD0045710|              2|
| RD0030877|              2|
| RD0002679|              2|
| RD0011169|              2|
| RD0028736|              2|
| RD0030669|              2|
| RD0061933|              2|
| RD0015904|              2|
| RD0008188|              2|
| RD0037899|              2|
| RD0000004|              2|
| RD0025628|              2|
| RD0048911|              2|
+----------+---------------+
only showing top 20 rows
Repeated ID Count: 700
Rows Before Deduplication: 102000
Unique Reading Count: 101300


In [28]:
pipeline_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).withColumn(
    "energy_category",
    F.when(F.col("energy_kwh") > 25, "High")
     .otherwise("Normal")
).groupBy(
    "energy_category"
).agg(
    F.sum("energy_kwh").alias("total_energy"),
    F.count("*").alias("total_readings")
).orderBy(
    F.col("total_energy").desc()
)
pipeline_df.explain(True)
pipeline_df.show()

== Parsed Logical Plan ==
'Sort ['total_energy DESC NULLS LAST], true
+- Aggregate [energy_category#968], [energy_category#968, sum(energy_kwh#20) AS total_energy#969, count(1) AS total_readings#970L]
   +- Project [reading_id#17, meter_id#18, reading_timestamp#19, energy_kwh#20, voltage_v#21, power_factor#22, outage_minutes#23, reading_status#24, source_type#25, ambient_temperature_c#26, operator_id#27, CASE WHEN (energy_kwh#20 > cast(25 as double)) THEN High ELSE Normal END AS energy_category#968]
      +- Filter (reading_status#24 = Valid)
         +- Relation [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] csv

== Analyzed Logical Plan ==
energy_category: string, total_energy: double, total_readings: bigint
Sort [total_energy#969 DESC NULLS LAST], true
+- Aggregate [energy_category#968], [energy_category#968, sum(energy_kwh#20) AS total_energy#969, 

In [30]:
import time
valid_readings_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).cache()
start_time = time.time()
count_first = valid_readings_df.count()
first_time = time.time() - start_time
start_time = time.time()
count_second = valid_readings_df.count()
second_time = time.time() - start_time
print("First Count:", count_first)
print("Second Count:", count_second)
print("First Execution Time:", first_time)
print("Second Execution Time:", second_time)
print("Is Cached:", valid_readings_df.is_cached)
valid_readings_df.unpersist()

First Count: 74486
Second Count: 74486
First Execution Time: 3.1025259494781494
Second Execution Time: 0.16208148002624512
Is Cached: True


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [31]:
outage_df = readings_df.filter(
    F.col("outage_minutes") > 0
).persist(
    StorageLevel.MEMORY_AND_DISK
)
# First action
print("Outage Count:", outage_df.count())
# Second action
outage_df.groupBy(
    "reading_status"
).count().show()
outage_df.unpersist()

Outage Count: 7653
+--------------+-----+
|reading_status|count|
+--------------+-----+
|      Rejected|  308|
|         Valid| 6345|
|     Estimated| 1000|
+--------------+-----+



DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [32]:
start_time = time.time()
readings_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
).count()
without_cache_time = time.time() - start_time
cached_df = readings_df.filter(
    F.col("reading_status") == "Valid"
).cache()
cached_df.count()
start_time = time.time()

cached_df.groupBy(
    "meter_id"
).agg(
    F.sum("energy_kwh").alias("total_energy")
).count()
with_cache_time = time.time() - start_time
print("Without Cache:", without_cache_time)
print("With Cache:", with_cache_time)
cached_df.unpersist()

Without Cache: 1.0248992443084717
With Cache: 0.4210166931152344


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [33]:
repartitioned_df = readings_df.repartition(8)
partition_count_df = repartitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
)
print("Number of Partitions:",
      repartitioned_df.rdd.getNumPartitions())
partition_count_df.show()

Number of Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11249|
|           1|11249|
|           2|11251|
|           3|11251|
|           4|11250|
|           5|11250|
|           6|11250|
|           7|11250|
+------------+-----+



In [34]:
twelve_partition_df = readings_df.repartition(12)
print("Original Partitions:",
      twelve_partition_df.rdd.getNumPartitions())
coalesce_df = twelve_partition_df.coalesce(3)
print("Coalesce Partitions:",
      coalesce_df.rdd.getNumPartitions())
coalesce_df.explain()
repartition_three_df = twelve_partition_df.repartition(3)
print("Repartition Partitions:",
      repartition_three_df.rdd.getNumPartitions())
repartition_three_df.explain()

Original Partitions: 12
Coalesce Partitions: 3
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2962]
            +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2962]
      +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_min

In [35]:
clean_readings_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() &
    F.col("voltage_v").isNotNull()
)
enriched_df = clean_readings_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).join(
    substation_df,
    on="substation_id",
    how="inner"
)
enriched_output_df = enriched_df.repartition(4)
enriched_output_df.write \
    .mode("overwrite") \
    .parquet("/content/energy_enriched_parquet")
print("Parquet output saved successfully")
enriched_output_df.show(10)
import os
print(os.listdir("/content/energy_enriched_parquet"))

Parquet output saved successfully
+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-------------------+-----------------+-----------------+------------+---------+-------+-----------+-----------------+
|substation_id|meter_id|reading_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|      site_type|          tariff_id|installation_date|rated_capacity_kw|meter_status|     city| region|capacity_mw|commissioned_year|
+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-------------------+-----------------+-----------------+------------+---------+-------+-----------+-----------------+
|        SUB09|   M0074| RD0048512|2026-02-05 11:00:00|     74.99|   

In [36]:
capstone_combined_df = readings_df.unionByName(
    extra_readings_df
)
print("Combined Rows:", capstone_combined_df.count())
capstone_unique_df = capstone_combined_df.dropDuplicates(
    ["reading_id"]
)
print("Unique Rows:", capstone_unique_df.count())

Combined Rows: 102000
Unique Rows: 101300


In [37]:
capstone_clean_df = capstone_unique_df.filter(
    F.col("energy_kwh").isNotNull() &
    (F.col("reading_status") != "Rejected")
)
print("Rows After Cleaning:", capstone_clean_df.count())
capstone_clean_df.show(10)

Rows After Cleaning: 95408
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|       0.831|             0|         Valid|     Manual|                 19.4|       NULL|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|    240.7

In [38]:
capstone_joined_df = capstone_clean_df.join(
    meter_df,
    on="meter_id",
    how="inner"
).join(
    substation_df,
    on="substation_id",
    how="inner"
).join(
    tariff_df,
    on="tariff_id",
    how="inner"
)
capstone_joined_df.show(10)

+-------------------+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-----------------+-----------------+------------+-------------+-------+-----------+-----------------+--------------------+---------------+----------------+------------------------+
|          tariff_id|substation_id|meter_id|reading_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|      site_type|installation_date|rated_capacity_kw|meter_status|         city| region|capacity_mw|commissioned_year|         tariff_name|  site_category|rate_per_kwh_inr|monthly_fixed_charge_inr|
+-------------------+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-----------------+-----------------

In [39]:
capstone_monthly_df = capstone_joined_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
).withColumn(
    "month",
    F.date_format("reading_timestamp", "yyyy-MM")
).withColumn(
    "energy_charge_inr",
    F.col("energy_kwh") * F.col("rate_per_kwh_inr")
)
monthly_report_df = capstone_monthly_df.groupBy(
    "region",
    "site_type",
    "month"
).agg(
    F.sum("energy_kwh").alias("total_kwh"),
    F.sum("energy_charge_inr").alias("total_energy_charges_inr")
)
monthly_report_df.show(20)

+-------+---------------+-------+------------------+------------------------+
| region|      site_type|  month|         total_kwh|total_energy_charges_inr|
+-------+---------------+-------+------------------+------------------------+
|Central|     Industrial|2026-08| 9938.689999999993|       90442.07899999994|
|  North|    Residential|2026-01|           2526.67|      15718.366000000002|
|  South|     Commercial|2026-02|35149.359999999986|       288224.7520000001|
|   West|     Industrial|2026-04| 33763.40999999999|              307247.031|
|   West|     Commercial|2026-04|          20686.28|       169627.4960000001|
|  North|     Industrial|2026-01|30649.829999999994|       278913.4530000001|
|  South|     Industrial|2026-08| 19867.48999999999|      180794.15899999999|
|  South|Public Facility|2026-07|           5904.88|               39562.696|
|   West|     Commercial|2026-02|          19985.61|      163882.00199999998|
|   West|     Commercial|2026-08|           7945.48|       65152

In [40]:
outage_report_df = capstone_monthly_df.groupBy(
    "region",
    "site_type",
    "month"
).agg(
    F.sum("outage_minutes").alias("total_outage_minutes"),

    F.count("*").alias("total_readings"),

    F.sum(
        F.when(F.col("outage_minutes") > 0, 1)
         .otherwise(0)
    ).alias("outage_readings")
).withColumn(
    "outage_percentage",
    F.round(
        F.col("outage_readings") /
        F.col("total_readings") * 100,
        2
    )
)
outage_report_df.show(20)

+-------+---------------+-------+--------------------+--------------+---------------+-----------------+
| region|      site_type|  month|total_outage_minutes|total_readings|outage_readings|outage_percentage|
+-------+---------------+-------+--------------------+--------------+---------------+-----------------+
|Central|     Industrial|2026-08|                1409|           167|             15|             8.98|
|  North|    Residential|2026-01|                5211|           691|             63|             9.12|
|  South|     Commercial|2026-02|               13751|          1829|            156|             8.53|
|   West|     Industrial|2026-04|                4858|           547|             48|             8.78|
|   West|     Commercial|2026-04|                8013|          1093|             90|             8.23|
|  North|     Industrial|2026-01|                3548|           518|             37|             7.14|
|  South|     Industrial|2026-08|                2921|          

In [41]:
final_report_df = monthly_report_df.join(
    outage_report_df,
    on=["region", "site_type", "month"],
    how="inner"
)
rank_window = Window.partitionBy(
    "region"
).orderBy(
    F.col("total_energy_charges_inr").desc()
)
final_report_df = final_report_df.withColumn(
    "site_type_rank",
    F.dense_rank().over(rank_window)
)
final_report_df.orderBy(
    "region",
    "site_type_rank",
    "month"
).show(50, truncate=False)

+-------+---------------+-------+------------------+------------------------+--------------------+--------------+---------------+-----------------+--------------+
|region |site_type      |month  |total_kwh         |total_energy_charges_inr|total_outage_minutes|total_readings|outage_readings|outage_percentage|site_type_rank|
+-------+---------------+-------+------------------+------------------------+--------------------+--------------+---------------+-----------------+--------------+
|Central|Industrial     |2026-01|24871.22          |226328.102              |4183                |418           |47             |11.24            |1             |
|Central|Industrial     |2026-05|24016.04          |218545.9640000001       |3191                |387           |35             |9.04             |2             |
|Central|Industrial     |2026-03|22963.22000000001 |208965.30199999997      |3331                |378           |38             |10.05            |3             |
|Central|Industrial   

In [42]:
cached_report_df = final_report_df.cache()
# First action
print("Total Report Rows:", cached_report_df.count())
# Second action
cached_report_df.show(20, truncate=False)
print("Is Cached:", cached_report_df.is_cached)
cached_report_df.unpersist()
print("Cache released successfully")

Total Report Rows: 120
+------+---------------+-------+------------------+------------------------+--------------------+--------------+---------------+-----------------+--------------+
|region|site_type      |month  |total_kwh         |total_energy_charges_inr|total_outage_minutes|total_readings|outage_readings|outage_percentage|site_type_rank|
+------+---------------+-------+------------------+------------------------+--------------------+--------------+---------------+-----------------+--------------+
|South |Industrial     |2026-03|54379.57000000001 |494854.08699999994      |6889                |875           |71             |8.11             |1             |
|South |Industrial     |2026-05|53718.419999999984|488837.622              |7731                |866           |82             |9.47             |2             |
|South |Industrial     |2026-06|52379.31          |476651.721              |6037                |854           |68             |7.96             |3             |
|Sout

In [43]:
final_output_df = final_report_df.repartition(2)
final_output_df.write \
    .mode("overwrite") \
    .parquet("/content/energy_monthly_final_report")
print("Final Monthly Energy Report saved successfully")
final_output_df.show(20, truncate=False)
import os
print(os.listdir("/content/energy_monthly_final_report"))

Final Monthly Energy Report saved successfully
+-------+---------------+-------+------------------+------------------------+--------------------+--------------+---------------+-----------------+--------------+
|region |site_type      |month  |total_kwh         |total_energy_charges_inr|total_outage_minutes|total_readings|outage_readings|outage_percentage|site_type_rank|
+-------+---------------+-------+------------------+------------------------+--------------------+--------------+---------------+-----------------+--------------+
|North  |Commercial     |2026-02|17607.859999999993|144384.45200000008      |6079                |928           |71             |7.65             |12            |
|Central|Industrial     |2026-01|24871.22          |226328.102              |4183                |418           |47             |11.24            |1             |
|North  |Commercial     |2026-04|18248.400000000005|149636.88               |7017                |972           |79             |8.13     